In [0]:
%run ./00_0_config

In [0]:
%sh pwd

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("sql_location", f"{paths['bronze']}mobile_measurements")

sql_location = dbutils.widgets.get("sql_location")

print(f"SQL lokacija je: {sql_location}")

In [0]:
%sql
select :sql_location as location_param;

In [0]:
raw_df = spark.read.json(paths['raw'])

raw_df.createOrReplaceTempView("tmp_raw_measurements")

spark.sql("""DROP TABLE IF EXISTS telco_prod.`01-bronze`.mobile_measurements""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS telco_prod.`01-bronze`.mobile_measurements
USING DELTA
PARTITIONED BY (processing_date)
LOCATION '{sql_location}'
AS
SELECT *, current_timestamp as _ingested_at
FROM tmp_raw_measurements
""")

In [0]:
display(spark.sql("DESCRIBE DETAIL telco_prod.`01-bronze`.mobile_measurements"))

In [0]:
display(spark.table("telco_prod.`01-bronze`.mobile_measurements"))